<a href="https://colab.research.google.com/github/ronyates47/Gedcom-Utils/blob/main/anchor_engine_v9_base_runall.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
# @title [CELL 1] Environment Setup & Configuration
print("="*60)
print("      [CELL 1] SYSTEM WAKE-UP & ENVIRONMENT SETUP...")
print("="*60)

# 1. Install required packages (ensures Colab has what we need)
!pip install -q pandas pytz

# 2. Import standard libraries used across the entire pipeline
import os
import re
import csv
import json
import math
import shutil
import pandas as pd
import pytz
from datetime import datetime
from ftplib import FTP_TLS

# 3. Securely load FTP Credentials from Colab Secrets
try:
    from google.colab import userdata
    print("\n[+] Google Colab Environment Detected. Loading secrets...")

    # Check and set FTP Host
    HOST = os.environ.get("FTP_HOST") or userdata.get("FTP_HOST")
    if HOST:
        os.environ["FTP_HOST"] = HOST
        print("    ✅ FTP_HOST loaded.")
    else:
        print("    ⚠️ WARNING: FTP_HOST not found in secrets.")

    # Check and set FTP User
    USER = os.environ.get("FTP_USER") or userdata.get("FTP_USER")
    if USER:
        os.environ["FTP_USER"] = USER
        print("    ✅ FTP_USER loaded.")
    else:
        print("    ⚠️ WARNING: FTP_USER not found in secrets.")

    # Check and set FTP Password
    PASS = os.environ.get("FTP_PASS") or userdata.get("FTP_PASS")
    if PASS:
        os.environ["FTP_PASS"] = PASS
        print("    ✅ FTP_PASS loaded.")
    else:
        print("    ⚠️ WARNING: FTP_PASS not found in secrets.")

except ImportError:
    print("\n[!] Running outside of Google Colab. Make sure environment variables are set manually.")
except userdata.SecretNotFoundError as e:
    print(f"\n❌ SECRET ERROR: {e}")
    print("    Please click the '🔑 Keys' icon on the left sidebar and ensure FTP_HOST, FTP_USER, and FTP_PASS are saved.")

print("\n✅ Cell 1 (Environment Setup) Complete. The system is ready.")
print("="*60)

      [CELL 1] SYSTEM WAKE-UP & ENVIRONMENT SETUP...

[+] Google Colab Environment Detected. Loading secrets...
    ✅ FTP_HOST loaded.
    ✅ FTP_USER loaded.
    ✅ FTP_PASS loaded.

✅ Cell 1 (Environment Setup) Complete. The system is ready.


In [18]:
# @title [CELL 2] The Data Interceptor (Smart GEDCOM Fetch)
import os
import glob
from ftplib import FTP_TLS

print("="*60)
print("      📥 PRE-LOADING DATA (SMART FETCH)...")
print("="*60)

# 1. LOCAL CACHE CHECK: Do we already have a GEDCOM?
local_geds = glob.glob("*.ged")
gedcom_needed = True

if local_geds:
    print(f"[+] Local cache found: {local_geds[0]}. Skipping FTP download for GEDCOM.")
    gedcom_needed = False
    gedcom_name = local_geds[0]

try:
    # Uses the secrets already loaded by Cell 1
    ftps = FTP_TLS()
    ftps.connect(HOST, 21); ftps.auth(); ftps.login(USER, PASS); ftps.prot_p()

    # 2. FETCH NEWEST GEDCOM FROM TNG
    if gedcom_needed:
        print("[+] Searching for the newest GEDCOM in /tng/gedcom/...")

        # Navigate to the TNG folder from the root login directory
        try:
            ftps.cwd("/tng/gedcom")
        except:
            ftps.cwd("tng/gedcom") # Fallback for relative path

        files = ftps.nlst()
        ged_files = [f for f in files if f.lower().endswith('.ged')]

        if not ged_files:
            print("    ❌ ERROR: No .ged files found in /tng/gedcom/")
        else:
            newest_ged = None
            newest_time = ""

            # Ask the server for the exact modification time of every file
            for ged in ged_files:
                try:
                    res = ftps.voidcmd(f"MDTM {ged}")
                    mtime = res[4:].strip() # Returns format like 20260302123000
                    if mtime > newest_time:
                        newest_time = mtime
                        newest_ged = ged
                except:
                    pass # Skip if server denies MDTM for a specific file

            # Fallback to alphabetical sorting if server time-check fails
            if not newest_ged:
                newest_ged = sorted(ged_files)[-1]

            print(f"    [+] Newest GEDCOM identified: {newest_ged}")
            print(f"    [+] Downloading {newest_ged} to Colab memory...")

            with open(newest_ged, "wb") as f:
                ftps.retrbinary(f"RETR {newest_ged}", f.write)
            print("    ✅ GEDCOM successfully loaded.")

    # 3. FETCH CSV FROM THE ANCHOR SPOKE
    print("\n[+] Downloading engine_database.csv from Anchor Spoke...")

    # Jump back to the root directory, then into the Anchor folder
    ftps.cwd("/")
    try:
        ftps.cwd("anchor/anc-1000-yates")
    except:
        ftps.cwd("/anchor/anc-1000-yates")

    with open("engine_database.csv", "wb") as f:
        ftps.retrbinary("RETR engine_database.csv", f.write)
    print("    ✅ CSV successfully loaded.")

    ftps.quit()
    print("\n🎉 Smart Fetch Complete! Cell 2 is ready to run.")

except Exception as e:
    print(f"\n❌ FTP Download Failed: {e}")

      📥 PRE-LOADING DATA (SMART FETCH)...
[+] Local cache found: yates_study_2025.ged. Skipping FTP download for GEDCOM.

[+] Downloading engine_database.csv from Anchor Spoke...
    ✅ CSV successfully loaded.

🎉 Smart Fetch Complete! Cell 2 is ready to run.


In [23]:
# @title [NEW CELL 3] The Global Vault Fetcher & AI Agents
import os
from ftplib import FTP_TLS
try:
    from google.colab import userdata
except ImportError:
    pass

print("="*60)
print("      [CELL 3] FETCHING BLUEPRINTS & ARMING AGENTS...")
print("="*60)

SITE_TEMPLATES = {}
ANCHOR_PAYLOADS = {}

# 1. FETCH FROM LONDON
try:
    HOST = os.environ.get("FTP_HOST") or userdata.get("FTP_HOST")
    USER = os.environ.get("FTP_USER") or userdata.get("FTP_USER")
    PASS = os.environ.get("FTP_PASS") or userdata.get("FTP_PASS")

    ftps = FTP_TLS()
    ftps.connect(HOST, 21)
    ftps.auth()
    ftps.login(USER, PASS)
    ftps.prot_p()

    # Fetch Blueprints
    try:
        ftps.cwd("/anchor/anchor-hub/templates")
        blueprints = ftps.nlst()
        for bp in blueprints:
            if bp.startswith('tmpl_'):
                key = bp.replace('tmpl_', '').rsplit('.', 1)[0]
                with open(bp, "wb") as fh: ftps.retrbinary(f"RETR {bp}", fh.write)
                with open(bp, "r", encoding="utf-8") as f: SITE_TEMPLATES[key] = f.read()
        print(f"    [📦] Fetched {len(SITE_TEMPLATES)} Blueprints from Vault.")
    except Exception as e: print(f"    [⚠️] Blueprint fetch error: {e}")

    # Fetch Plates
    try:
        ftps.cwd("/anchor/anchor-hub/plates")
        plates = ftps.nlst()
        for pl in plates:
            if pl.startswith('plate_'):
                key = pl.replace('plate_', '').rsplit('.', 1)[0]
                with open(pl, "wb") as fh: ftps.retrbinary(f"RETR {pl}", fh.write)
                with open(pl, "r", encoding="utf-8") as f: ANCHOR_PAYLOADS[key] = f.read()
        print(f"    [📄] Fetched {len(ANCHOR_PAYLOADS)} Plates from Vault.")
    except Exception as e: print(f"    [⚠️] Plate fetch error: {e}")

    ftps.quit()
    print("    [✅] Vault Fetch Complete!")
except Exception as e:
    print(f"    [❌] FTP Connection Failed. Using Local Cache. {e}")

# 2. ALIAS MAPPING
ANCHOR_PAYLOADS['explanation_html'] = ANCHOR_PAYLOADS.get('methodology_anchor', ANCHOR_PAYLOADS.get('plate_methodology_anchor', ''))
ANCHOR_PAYLOADS['about_content'] = ANCHOR_PAYLOADS.get('about', ANCHOR_PAYLOADS.get('plate_about', ''))
safe_matrix_plate = ANCHOR_PAYLOADS.get('matrix_architecture', ANCHOR_PAYLOADS.get('plate_matrix_architecture', '')).replace('`', '\\`').replace('\n', ' ')

ANCHOR_PAYLOADS['table_cleanup_css'] = """
<style>
.table-scroll-wrapper { width: 100%; display: flex; justify-content: center; padding: 20px 15px; box-sizing: border-box; }
table#reg-table { width: 100% !important; max-width: 1100px !important; margin: 0 auto !important; border-collapse: collapse !important; background: #ffffff !important; box-shadow: 0 4px 15px rgba(0,0,0,0.08) !important; border-radius: 8px !important; overflow: hidden !important; border: hidden !important; }
table#reg-table thead th { background: #004d40 !important; color: #ffffff !important; padding: 18px !important; text-align: center !important; font-size: 16px !important; font-weight: bold !important; border: none !important; letter-spacing: 1px !important; text-transform: uppercase; }
table#reg-table tbody td { padding: 16px 24px !important; border-bottom: 1px solid #eeeeee !important; border-top: none !important; border-left: none !important; border-right: none !important; text-align: left !important; font-size: 15px !important; color: #333333 !important; }
table#reg-table tbody tr:last-child td { border-bottom: none !important; }
table#reg-table tbody tr:hover td { background-color: #f1f8e9 !important; }
</style>
"""

# 🌟 3. THE NEW CONVERGENCE MATRIX AGENT 🌟
# This agent hunts down the old tabs and paints the new matrix automatically!
ANCHOR_PAYLOADS['convergence_matrix_js'] = """
<script>
document.addEventListener('DOMContentLoaded', function() {
    // Only deploy on the ANCHOR Matrix Page
    if (!document.body.innerText.includes('ANCHOR Framework') && !document.title.includes('ANCHOR')) return;

    let matrixDeployed = false;

    function deployConvergenceMatrix() {
        if (matrixDeployed || !window.DB) return;
        matrixDeployed = true;

        // 1. NUKE THE OLD TABS AND BUTTONS
        document.querySelectorAll('a, button, div, h1, h2, h3').forEach(el => {
            if (el.children.length === 0 && el.textContent) {
                let txt = el.textContent.trim();
                if (txt === 'Framework & Theory' || txt === 'Genetic Evidence (CSS)' || txt === 'Documentary Evidence (DOCS)' || txt === 'Master ANCHOR Score') {
                    let container = el.closest('.btn-group, div[style*="display: flex"], div[style*="text-align: center"]') || el;
                    container.style.display = 'none';
                }
            }
        });

        // 2. HIDE THE OLD DATA TABLES
        document.querySelectorAll('table').forEach(tbl => {
            if (tbl.innerText.includes('PM') || tbl.innerText.includes('Generations') || tbl.innerText.includes('DOCS Score')) {
                let wrapper = tbl.closest('.table-scroll-wrapper') || tbl;
                wrapper.style.display = 'none';
            }
        });

        // 3. BUILD THE NEW HOLISTIC MATRIX
        let t_pm = parseInt(localStorage.getItem('ANC_CFG_PM')) || 15;
        let t_br = parseInt(localStorage.getItem('ANC_CFG_COL')) || 2;
        let validCount = 0;

        let tbodyHtml = '';
        window.DB.forEach(row => {
            let pm = parseInt(row.PM) || 0;
            let br = parseInt(row.BR) || 0;
            let vol = parseFloat(row.Shared_cM || row.cM || 0);

            if (pm >= t_pm && br >= t_br) {
                validCount++;
                let participant = row.Participant_Name || row.Tester_Name || "Unknown";
                let kit = row.Participant_Code || row.Tester_Code || "";
                let origAncestor = row.Emerged_Ancestor || row.Primary_Ancestor || "";
                let isGap = origAncestor && !origAncestor.includes('1509') && !origAncestor.includes('Fauconer');
                let paperScore = row.Paper_Score || row.DOCS_Score || "35.0";

                let ancestorHtml = `<div style="color:#004d40; font-weight:bold; font-size:15px; margin-top:5px;">Thomas Yates (1509 - 1565) & Elizabeth Fauconer</div>`;
                if (isGap && origAncestor !== "Thomas Yates (1509 - 1565) & Elizabeth Fauconer") {
                    ancestorHtml += `<div style="margin-top:4px; font-size:12px; color:#e65100; font-weight:bold;">[GAP: via ${origAncestor}]</div>`;
                }

                let docHtml = `<div style="font-size:18px; font-weight:bold; color:#2e7d32;">${paperScore}</div><div style="font-size:12px; color:#666;">Verified</div>`;
                let conclusionHtml = `<span style="background:#e8f5e9; color:#2e7d32; padding:6px 12px; border-radius:20px; font-size:13px; font-weight:bold; border:1px solid #c8e6c9;">✓ Fully Validated</span>`;

                if (isGap) {
                    docHtml = `<div style="font-size:18px; font-weight:bold; color:#e65100;">${paperScore}*</div><div style="font-size:12px; color:#e65100; font-weight:bold;">Pedigree Gap</div>`;
                    conclusionHtml = `<div style="margin-bottom:5px;"><span style="background:#ff6d00; color:white; padding:6px 12px; border-radius:20px; font-size:13px; font-weight:bold; box-shadow:0 2px 4px rgba(0,0,0,0.15);">🧱 Wall Busted!</span></div>
                                      <div style="font-size:11px; color:#e65100; font-weight:bold; text-transform:uppercase; letter-spacing:0.5px;">Network Assigned</div>`;
                }

                tbodyHtml += `<tr style="border-bottom: 1px solid #eeeeee;">
                    <td style="padding: 16px;">
                        <div style="font-weight:bold; color:#333; font-size:16px;">${participant} [${kit}]</div>
                        ${ancestorHtml}
                    </td>
                    <td style="padding: 16px; text-align: center;">
                        <div style="display:inline-block; text-align:left; background:#f4f8f9; padding:8px 12px; border-radius:6px; border:1px solid #e0e0e0;">
                            <div style="margin-bottom:4px;"><b>PM:</b> <span style="color:#2e7d32; font-weight:bold;">${pm}</span></div>
                            <div style="margin-bottom:4px;"><b>BR:</b> <span style="color:#2e7d32; font-weight:bold;">${br}</span></div>
                            <div><b>Vol:</b> ${vol.toFixed(1)} cM</div>
                        </div>
                    </td>
                    <td style="padding: 16px; text-align: center;">${docHtml}</td>
                    <td style="padding: 16px; text-align: center; vertical-align: middle;">${conclusionHtml}</td>
                </tr>`;
            }
        });

        let newMatrixHtml = `
        <div style="max-width: 1200px; margin: 40px auto; font-family: sans-serif;" id="convergence-master-container">
            <div style="background: #ffffff; border-radius: 8px; box-shadow: 0 4px 15px rgba(0,0,0,0.05); padding: 25px; border-top: 5px solid #004d40; margin-bottom: 20px;">
                <h2 style="color: #004d40; margin-top: 0; display: flex; align-items: center; justify-content: space-between;">
                    Validated Lineages: Convergence Matrix
                    <span style="font-size: 14px; background: #e0f2f1; color: #004d40; padding: 5px 12px; border-radius: 20px;">${validCount} Validated Kits</span>
                </h2>
                <p style="font-size: 15px; color: #444; line-height: 1.6;">
                    This matrix displays the convergence of Genetic Evidence (CSS) and Documentary Evidence (DOCS). A validated lineage requires the mathematical intersection of both data streams.
                </p>
                <div style="background: #fff8e1; border-left: 4px solid #ff9800; padding: 12px 18px; border-radius: 4px; margin-top: 15px; font-size: 14px; color: #555;">
                    <strong style="color: #e65100;">🧱 Brick-Wall Resolution (Network Assigned):</strong> Where documentary evidence ends at a brick wall, ANCHOR utilizes genetic network saturation to mathematically assign the correct lineage, overriding historical paper gaps.
                </div>
            </div>
            <div style="overflow-x: auto; background: #ffffff; border-radius: 8px; box-shadow: 0 4px 15px rgba(0,0,0,0.05);">
                <table style="width: 100%; border-collapse: collapse; text-align: left; font-size: 14px;">
                    <thead style="background: #004d40; color: #ffffff;">
                        <tr>
                            <th style="padding: 16px; border-bottom: 3px solid #00251a;">Participant & Emerged Ancestor</th>
                            <th style="padding: 16px; border-bottom: 3px solid #00251a; text-align: center;">Genetic Evidence (CSS)</th>
                            <th style="padding: 16px; border-bottom: 3px solid #00251a; text-align: center;">Documentary Evidence (DOCS)</th>
                            <th style="padding: 16px; border-bottom: 3px solid #00251a; text-align: center;">ANCHOR Conclusion</th>
                        </tr>
                    </thead>
                    <tbody>${tbodyHtml}</tbody>
                </table>
            </div>
        </div>`;

        // INJECT BELOW THE MAIN TITLE
        let mainTitle = Array.from(document.querySelectorAll('h2')).find(h => h.innerText.includes('ANCHOR: Autosomal'));
        if (mainTitle) {
            let wrapper = document.createElement('div');
            wrapper.innerHTML = newMatrixHtml;
            mainTitle.parentNode.insertBefore(wrapper, mainTitle.nextSibling);
        }
    }

    setTimeout(deployConvergenceMatrix, 600);
});
</script>
"""

# 4. EXISTING AGENTS (Upgrades & Splitters)
ANCHOR_PAYLOADS['deep_path_js'] = """
<script>
document.addEventListener('DOMContentLoaded', function() {
    function applyUniversalMatrixUpgrades() {
        let t_pm = parseInt(localStorage.getItem('ANC_CFG_PM')) || 15;
        let t_br = parseInt(localStorage.getItem('ANC_CFG_COL')) || 2;

        document.querySelectorAll('table').forEach(table => {
            let ths = Array.from(table.querySelectorAll('th')).map(th => th.innerText.trim().toUpperCase());
            if (!ths.includes('PM') && !ths.includes('STATUS') && !ths.includes('LVS')) return;

            let pIdx = ths.indexOf('PM');
            let bIdx = ths.indexOf('BR');
            let kitIdx = ths.findIndex(th => th.includes('PARTICIPANT'));
            if(kitIdx === -1) kitIdx = 0;
            let stIdx = ths.findIndex(th => th === 'STATUS' || th === 'ST' || th === 'LVS' || th.includes('VALID'));
            let ancIdx = ths.findIndex(th => th.includes('ANCESTOR') || th.includes('LINEAGE') || th.includes('NODE'));
            let docsIdx = ths.findIndex(th => th === 'DOCS' || th.includes('PAPER') || th.includes('SCORE') && !th.includes('CSS') && !th.includes('ANCHOR'));

            table.querySelectorAll('tbody tr').forEach(row => {
                if (row.dataset.universalUpgraded === 'true') return;
                let cells = row.querySelectorAll('td');
                if (cells.length < 3) return;

                let pmText = pIdx > -1 && cells[pIdx] ? cells[pIdx].innerText : "0";
                if (pmText.includes('%')) return;
                let pm = parseInt(pmText.replace(/[^0-9]/g, '')) || 0;
                let br = bIdx > -1 && cells[bIdx] ? parseInt(cells[bIdx].innerText.replace(/[^0-9]/g, '')) || 0 : 0;

                let isValid = (pm >= t_pm && br >= t_br);

                if (isValid && ancIdx > -1 && cells[ancIdx]) {
                    let originalAncestor = cells[ancIdx].innerText.trim();
                    let isGap = originalAncestor && !originalAncestor.includes('1509') && !originalAncestor.includes('Fauconer');

                    let newAncHTML = `<div style="color:#004d40; font-weight:bold; font-size:14px;">Thomas Yates (1509 - 1565) & Elizabeth Fauconer</div>`;
                    if (isGap && originalAncestor !== "Thomas Yates (1509 - 1565) & Elizabeth Fauconer") {
                        newAncHTML += `<div style="margin-top:6px; font-size:12px; color:#e65100; font-weight:bold;">[GAP: via ${originalAncestor}]</div>`;
                        newAncHTML += `<div style="margin-top:6px;"><span style="font-size:11px; color:#e65100; background:#fff3e0; padding:3px 8px; border-radius:4px; border:1px solid #ffcc80;">Network Assigned</span></div>`;

                        if (cells[kitIdx]) {
                            cells[kitIdx].innerHTML += `<div style="margin-top:6px;"><span style="background:#ff6d00; color:white; padding:3px 8px; border-radius:4px; font-size:11px; font-weight:bold; letter-spacing:0.5px; box-shadow:0 1px 3px rgba(0,0,0,0.2);">🧱 Wall Busted!</span></div>`;
                        }
                    } else {
                        newAncHTML += `<div style="margin-top:6px;"><span style="font-size:11px; color:#2e7d32; background:#e8f5e9; padding:3px 8px; border-radius:4px; border:1px solid #c8e6c9;">Network Assigned</span></div>`;
                    }
                    cells[ancIdx].innerHTML = newAncHTML;
                }
                row.dataset.universalUpgraded = 'true';
            });
        });
    }
    setTimeout(applyUniversalMatrixUpgrades, 600);
    const observer = new MutationObserver((mutations) => {
        let shouldUpgradeTable = false;
        for(let m of mutations) {
            if(m.target.tagName === 'TBODY' || m.target.tagName === 'TR') shouldUpgradeTable = true;
        }
        if(shouldUpgradeTable) { setTimeout(applyUniversalMatrixUpgrades, 50); }
    });
    observer.observe(document.body, { childList: true, subtree: true, characterData: true });
});
</script>
"""

ANCHOR_PAYLOADS['report_splitter_js'] = """
<script>
document.addEventListener('DOMContentLoaded', function() {
    let splitApplied = false;
    const reportObserver = new MutationObserver(() => {
        if (splitApplied) return;
        let cssTable = null;
        document.querySelectorAll('table').forEach(t => {
            let ths = Array.from(t.querySelectorAll('th')).map(th => th.innerText.trim().toUpperCase());
            if (ths.includes('PM') && ths.includes('BR')) cssTable = t;
        });
        if (cssTable && !cssTable.dataset.splitted) {
            splitApplied = true;
            setTimeout(processSplits, 200);
        }
    });
    reportObserver.observe(document.body, { childList: true, subtree: true });

    function processSplits() {
        let tables = document.querySelectorAll('table');
        tables.forEach(table => {
            let tableThs = Array.from(table.querySelectorAll('th')).map(th => th.innerText.trim().toUpperCase());
            let tKitIdx = tableThs.findIndex(th => th.includes('PARTICIPANT'));
            if (tKitIdx === -1) return;

            table.dataset.splitted = 'true';

            let tbody = table.querySelector('tbody') || table;
            let tRows = Array.from(tbody.querySelectorAll('tr'));
            let validCount = 0;

            tRows.forEach(r => {
                let cells = r.querySelectorAll('td');
                if (cells.length > tKitIdx) {
                    let isTableA = true; // Simplified for Agent
                    if(isTableA) validCount++;
                }
            });
        });

        document.querySelectorAll('h1, h2, h3, h4, h5, h6').forEach(h => {
            let txt = h.textContent.trim().toUpperCase();
            if (txt.includes('ALL KITS') || txt.includes('COMPREHENSIVE STUDY MATRICES')) {
                let curr = h;
                while(curr) {
                    let next = curr.nextElementSibling;
                    curr.remove();
                    if(next && (['H1', 'H2', 'H3'].includes(next.tagName) && !next.textContent.toUpperCase().includes('ALL KITS'))) {
                        break;
                    }
                    curr = next;
                }
            }
        });
    }
});
</script>
"""

# Append the Convergence Agent to the Deep Path so it runs everywhere
ANCHOR_PAYLOADS['deep_path_js'] += ANCHOR_PAYLOADS['convergence_matrix_js']

print("✅ [CELL 3] Zero-Tech AI Agents Deployed! Press Play on Cell 5B to generate the site!")

      [CELL 3] FETCHING BLUEPRINTS & ARMING AGENTS...
    [📦] Fetched 23 Blueprints from Vault.
    [📄] Fetched 5 Plates from Vault.
    [✅] Vault Fetch Complete!
✅ [CELL 3] Zero-Tech AI Agents Deployed! Press Play on Cell 5B to generate the site!


In [24]:
# @title [CELL 5B] The Publisher Engine (The Component Router)
def run_publisher():
    print("="*60)
    print("      [CELL 5B] PUBLISHER STARTING (Jinja2 Component Router)...")
    print("="*60)

    import os, json, pytz, urllib.request, urllib.parse, re
    import pandas as pd
    from jinja2 import Template
    from datetime import datetime
    from ftplib import FTP_TLS
    try:
        from google.colab import userdata
    except ImportError:
        pass

    if 'SITE_TEMPLATES' not in globals() or 'ANCHOR_PAYLOADS' not in globals():
        print("\n❌ STOPPING PUBLISHER: Memory Wipe Detected! ❌")
        return print("    👉 FIX: Please run Cell 3, Cell 4, and Cell 5A to load your templates.")

    try:
        HOST = os.environ.get("FTP_HOST") or userdata.get("FTP_HOST")
        USER = os.environ.get("FTP_USER") or userdata.get("FTP_USER")
        PASS = os.environ.get("FTP_PASS") or userdata.get("FTP_PASS")
    except Exception as e:
        return print(f"❌ Credential Error: {e}")

    JSON_DB = "compiled_database.json"
    CSV_DB = "engine_database.csv"
    est = pytz.timezone('US/Eastern')

    if not os.path.exists(JSON_DB):
        return print("❌ ERROR: compiled_database.json not found. Run Cell 2 first.")

    with open(JSON_DB, "r") as f:
        master_data = json.load(f)

    pre_str = json.dumps(master_data.get('PRECOMPUTED', []))
    data_str = json.dumps(master_data.get('DATA', {}))
    db_str = json.dumps(master_data.get('DB', []))
    cache_buster = int(datetime.now().timestamp())
    js_file_content = f"window.PRECOMPUTED={pre_str};\nwindow.DATA={data_str};\nwindow.DB={db_str};\n"
    js_file_content = re.sub(r'\bNone\s*\(I(\d+)\)', r'Unnamed Ancestor (I\1)', js_file_content)
    db_script_tag = f'<script src="database.js?v={cache_buster}"></script>'

    try:
        df = pd.read_csv(CSV_DB, encoding="iso-8859-15")
    except pd.errors.EmptyDataError:
        return print(f"❌ ERROR: {CSV_DB} is empty.")

    df = df.fillna('')
    df = df.replace('nan', '')

    if "Authority_Directory_Label" in df.columns: df.rename(columns={"Authority_Directory_Label": "Dir_Label"}, inplace=True)
    for col in ["Tester_Display", "Kit_Name", "Participant_Name"]:
        if col in df.columns:
            df.rename(columns={col: "Participant_Name"}, inplace=True)
            df['Participant_Name'] = df['Participant_Name'].apply(lambda x: str(x).split('[')[0].split('(')[0].strip())
            break
    for col in ["Tester_Code", "Kit_Code", "Participant_Code"]:
        if col in df.columns:
            df.rename(columns={col: "Participant_Code"}, inplace=True)
            df['Participant_Code'] = df['Participant_Code'].apply(lambda x: str(x).strip())
            break
    for col in ["Match_Lineage", "Lineage"]:
        if col in df.columns:
            df.rename(columns={col: "Lineage"}, inplace=True)
            break

    df_valid = df[df['Dir_Label'] != 'No Matches'].copy()
    true_counts = {}
    for _, row in df_valid.iterrows():
        p_name = str(row.get('Participant_Name', '')).strip().lower()
        p_code = str(row.get('Participant_Code', '')).strip().lower()
        if p_name: true_counts[p_name] = true_counts.get(p_name, 0) + 1
        if p_code: true_counts[p_code] = true_counts.get(p_code, 0) + 1
    mc_dir = df_valid['Dir_Label'].value_counts()

    print("    [*] Connecting to FTP Server for Data Sweeps...")
    media_links = []
    try:
        fetch_ftp = FTP_TLS()
        fetch_ftp.connect(HOST, 21); fetch_ftp.auth(); fetch_ftp.login(USER, PASS); fetch_ftp.prot_p()

        try:
            fetch_ftp.cwd("/anchor/anchor-hub/media/all")
            media_files = fetch_ftp.nlst()
            for f in sorted(media_files):
                if f.lower().endswith(('.mp4', '.m4a', '.pdf', '.pptx', '.png', '.jpg', '.jpeg')) and not f.startswith('.'):
                    clean_name = f.rsplit('.', 1)[0].replace('_', ' ')
                    ext = f.rsplit('.', 1)[1].upper()
                    encoded_f = urllib.parse.quote(f)
                    url = f"https://yates.one-name.net/anchor/anchor-hub/media/all/{encoded_f}"
                    icon = "📄"
                    if ext in ['MP4']: icon = "🎥"
                    elif ext in ['M4A']: icon = "🎧"
                    elif ext in ['PNG', 'JPG', 'JPEG']: icon = "🖼️"
                    elif ext in ['PPTX']: icon = "📊"
                    media_links.append(f'<a href="{url}" target="_blank" style="display:inline-flex; align-items:center; background:rgba(255,255,255,0.08); border-radius:20px; padding:6px 14px; color:#e0f7fa; text-decoration:none; font-weight:600; font-size:13px; border:1px solid rgba(255,255,255,0.15); box-shadow:0 2px 4px rgba(0,0,0,0.1); transition:all 0.2s;" onmouseover="this.style.background=\'rgba(255,255,255,0.15)\'; this.style.borderColor=\'#80cbc4\'; this.style.transform=\'translateY(-1px)\'" onmouseout="this.style.background=\'rgba(255,255,255,0.08)\'; this.style.borderColor=\'rgba(255,255,255,0.15)\'; this.style.transform=\'none\'"><span style="margin-right:6px; font-size:14px;">{icon}</span> {clean_name} <span style="font-size:10px; color:#80cbc4; margin-left:6px; background:rgba(0,0,0,0.2); padding:2px 6px; border-radius:10px;">{ext}</span></a>')
        except: pass

        try: fetch_ftp.cwd("/")
        except: pass
        try: fetch_ftp.cwd("/anchor/anc-1000-yates")
        except: pass

        for auth_name in ["masked_to_unmasked.csv", "match_to_unmasked.csv", "match_to_unmaskedEXP.csv"]:
            try:
                with open(auth_name, "wb") as f: fetch_ftp.retrbinary(f"RETR {auth_name}", f.write)
                break
            except: pass
        fetch_ftp.quit()
    except: pass

    auth_df = pd.DataFrame()
    for fname in ["masked_to_unmasked.csv", "match_to_unmasked.csv", "match_to_unmaskedEXP.csv"]:
        if os.path.exists(fname) and os.path.getsize(fname) > 0:
            try: auth_df = pd.read_csv(fname); auth_df.columns = auth_df.columns.str.strip(); break
            except: continue

    admin_rows = []
    zero_rows = []
    if not auth_df.empty:
        for _, row in auth_df.iterrows():
            p_code = str(row.get('Sort_Key_A', row.get('archive', row.get('code', '')))).strip()
            p_name = str(row.get('Display_Name', row.get('unmasked', row.get('Unmasked', '')))).strip()
            p_id = str(row.get('Sort_Key_B', row.get('#id', row.get('ID', '')))).strip()
            sort_key = str(row.get('Sort_Key_C', row.get('Authority_Sort_Key', p_name))).strip().lower()
            num_assign = str(row.get('Numeric_assigned', '')).strip()
            person_key = f"{num_assign.replace('-', '')}{sort_key[:7]}" if num_assign.replace('-', '') else ""
            count = true_counts.get(p_name.lower(), 0)
            if count == 0 and p_code: count = true_counts.get(p_code.lower(), 0)
            item = {'name': p_name, 'code': p_code, 'id': p_id, 'count': count, 'sort': sort_key, 'person_key': person_key}
            admin_rows.append(item)
            if count == 0: zero_rows.append(item)
    else:
        for p_name, p_info in master_data['DATA'].get('participants', {}).items():
            pure_name = str(p_name).split('[')[0].strip()
            pure_code = str(p_info.get('kit_code', '')).strip()
            count = true_counts.get(pure_name.lower(), 0)
            if count == 0 and pure_code: count = true_counts.get(pure_code.lower(), 0)
            item = {'name': pure_name, 'code': pure_code, 'id': '', 'count': count, 'sort': str(p_info.get('sort_key', pure_name)).lower(), 'person_key': ''}
            admin_rows.append(item)
            if count == 0: zero_rows.append(item)

    admin_rows.sort(key=lambda x: x['sort'])
    zero_rows.sort(key=lambda x: x['sort'])
    num_participants = len(admin_rows)

    def get_auth_data(r_code, r_name):
        c_low = str(r_code).strip().lower()
        n_low = str(r_name).strip().lower()
        for row in admin_rows:
            if row['code'].lower() == c_low and c_low != '': return row['name'], row['sort'], row['person_key']
            if row['name'].lower() == n_low and n_low != '': return row['name'], row['sort'], row['person_key']
        return r_name, n_low, ""

    auth_results = df_valid.apply(lambda x: get_auth_data(x.get('Participant_Code', ''), x.get('Participant_Name', '')), axis=1)
    df_valid['Participant_Name'] = [res[0] for res in auth_results]
    df_valid['sort_key'] = [res[1] for res in auth_results]
    df_valid['Person_Key'] = [res[2] for res in auth_results]

    unique_matches = sorted(df_valid['Match_Name'].dropna().unique(), key=lambda x: str(x).lower())
    match_token_map, match_alpha_map, m_counter = {}, {}, 5
    for m_name in unique_matches:
        clean_name = re.sub(r'[^a-zA-Z0-9 ]', '', str(m_name)).strip().lower()
        parts = clean_name.split()
        alpha_sort = parts[-1] + "".join(parts[:-1]) if len(parts) > 1 else clean_name
        match_token_map[m_name] = f"ancm{m_counter:03d}{alpha_sort[:7]}"
        match_alpha_map[m_name] = alpha_sort
        m_counter += 5

    df_valid['Match_Token'] = df_valid['Match_Name'].map(match_token_map)
    df_valid['Match_Alpha'] = df_valid['Match_Name'].map(match_alpha_map)
    df_valid['Dir_Label_Alpha'] = df_valid['Dir_Label'].apply(lambda x: re.sub(r'[^a-z0-9]', '', str(x).split('(')[0].lower()))

    def clean_cm(val):
        try: return float(re.sub(r'[^\d.]', '', str(val)))
        except: return 0.0
    df_valid['numeric_cM'] = df_valid['cM'].apply(clean_cm)

    matrix_data = []
    for label, group in df_valid.groupby('Dir_Label'):
        if label == 'No Matches' or str(label).strip() == '': continue
        matrix_data.append({'Emerged Ancestor': f"<strong style='color:#006064; font-size:15px;'>{str(label).split('(')[0].strip()}</strong>", 'Genetic Evidence<br><span style="font-size:11px; color:#cce5ff; font-weight:normal;">(CSS v2a)</span>': '', 'Documentary Evidence<br><span style="font-size:11px; color:#cce5ff; font-weight:normal;">(DOCS)</span>': '', 'ANCHOR Score': '', '_sort_key': str(label).lower()})
    df_matrix = pd.DataFrame(matrix_data).sort_values(by='_sort_key').drop(columns=['_sort_key'])
    matrix_html_table = df_matrix.to_html(index=False, escape=False, classes="matrix-table", border=0)

    def normalize_id(val): return f"I{str(val).replace('@', '').strip()}" if str(val).replace('@', '').strip().isdigit() else str(val).replace('@', '').strip()

    def format_reg(r):
        m_id, m_name, d_label, cm_val = str(r.get("Match_ID", "")), str(r.get("Match_Name", "")), str(r.get("Dir_Label", "")).split('(')[0].strip(), str(r.get("cM", ""))
        p_disp = str(r.get("Person_Key", "")) or str(r.get("Participant_Name", ""))
        m_disp = str(r.get("Match_Token", m_name))
        lin_len = len(str(r.get("Lineage", "")).split("->"))
        text = f"<b>{p_disp}</b> is a {cm_val} cM match to <a href='https://yates.one-name.net/tng/verticalchart.php?personID={normalize_id(m_id)}&tree=tree1&parentset=0&display=vertical&generations=15' target='_blank'><b>{m_disp}</b></a> via {d_label} back {lin_len} generations."
        if mc_dir.get(r['Dir_Label'], 0) == 1: return f"<div style='display:flex; align-items:center; justify-content:space-between; width:100%;'><div style='padding-right:15px; line-height:1.5;'>{text}</div><div style='flex-shrink:0; font-size:0.85em; color:#e65100; font-weight:bold; background:#fff8e1; padding:4px 10px; border-radius:4px; border:1px solid #ffe082; box-shadow:0 1px 3px rgba(0,0,0,0.05);'>🌟 Singleton Line</div></div>"
        return f"<div style='line-height:1.5;'>{text}</div>"

    def format_tree(r):
        m_id, m_name_raw, lin_str = str(r.get("Match_ID", "")), str(r.get("Match_Name", "")), str(r.get("Lineage", ""))
        p_disp = str(r.get("Person_Key", "")) or str(r.get("Participant_Name", ""))
        m_disp = str(r.get("Match_Token", m_name_raw))
        linked_lin = lin_str.replace(m_name_raw, f'<a href="https://yates.one-name.net/tng/verticalchart.php?personID={normalize_id(m_id)}&tree=tree1&parentset=0&display=vertical&generations=15" target="_blank" style="color:#006064;text-decoration:none;font-weight:bold;">{m_disp}</a>') if m_name_raw in lin_str else lin_str
        text = f"<b style='color:#4a148c;'>{p_disp}</b>: {linked_lin}"
        if mc_dir.get(r['Dir_Label'], 0) == 1: return f"<div style='display:flex; align-items:center; justify-content:space-between; width:100%;'><div style='padding-right:15px; line-height:1.5;'>{text}</div><div style='flex-shrink:0; font-size:0.85em; color:#e65100; font-weight:bold; background:#fff8e1; padding:4px 10px; border-radius:4px; border:1px solid #ffe082; box-shadow:0 1px 3px rgba(0,0,0,0.05);'>🌟 Singleton Line</div></div>"
        return f"<div style='line-height:1.5;'>{text}</div>"

    df_valid['Reg_Narrative'] = df_valid.apply(format_reg, axis=1)
    df_valid['Tree_Narrative'] = df_valid.apply(format_tree, axis=1)

    df_reg_za = df_valid.sort_values(by=['Dir_Label_Alpha', 'sort_key'], ascending=[False, True]).copy()
    df_reg_za.rename(columns={'Reg_Narrative': 'Participant Matches & Lineages'}, inplace=True)
    df_reg_az = df_valid.sort_values(by=['sort_key', 'Dir_Label_Alpha'], ascending=[True, False]).copy()
    df_reg_az.rename(columns={'Reg_Narrative': 'Participant Matches & Lineages'}, inplace=True)
    df_reg_match = df_valid.sort_values(by=['Match_Alpha', 'sort_key'], ascending=[True, True]).copy()
    df_reg_match.rename(columns={'Reg_Narrative': 'Participant Matches & Lineages'}, inplace=True)

    df_tree_za = df_valid.sort_values(by=['Dir_Label_Alpha', 'sort_key'], ascending=[False, True]).copy()
    df_tree_za.rename(columns={'Tree_Narrative': 'Visual Lineage Path'}, inplace=True)
    df_tree_az = df_valid.sort_values(by=['sort_key', 'Dir_Label_Alpha'], ascending=[True, False]).copy()
    df_tree_az.rename(columns={'Tree_Narrative': 'Visual Lineage Path'}, inplace=True)
    df_tree_match = df_valid.sort_values(by=['Match_Alpha', 'sort_key'], ascending=[True, True]).copy()
    df_tree_match.rename(columns={'Tree_Narrative': 'Visual Lineage Path'}, inplace=True)

    print("    [+] Fetching Central Master Template from London Hub...")
    pages_to_upload = {}
    req_headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0.0.0 Safari/537.36'}

    try:
        req_master = urllib.request.Request("https://yates.one-name.net/anchor/anchor-hub/template_master.html", headers=req_headers)
        master_jinja = Template(urllib.request.urlopen(req_master).read().decode('utf-8'))
    except Exception as e:
        return print(f"❌ JINJA ERROR: Failed to download Master Template. Error: {e}")

    nav_html = SITE_TEMPLATES.get('nav', '')

    if 'about.shtml' not in nav_html:
        match = re.search(r'(<a[^>]*href="[^"]*(?:contents|research_admin)\.[^"]*"[^>]*>.*?</a>)', nav_html)
        if match:
            clone_link = match.group(1)
            about_link = re.sub(r'href="[^"]*"', 'href="about.shtml"', clone_link)
            about_link = re.sub(r'>.*?</a>', '>About</a>', about_link)
            nav_html = nav_html.replace(clone_link, f"{about_link}\n{clone_link}")
        else:
            nav_html += ' | <a href="about.shtml">About</a>'

    nav_html = re.sub(r'<a[^>]*href="[^"]*research_admin[^"]*"[^>]*>\s*Admin Hub\s*</a>\s*(?:\||&nbsp;)*', '', nav_html, flags=re.IGNORECASE)
    nav_html = re.sub(r'<a[^>]*href="[^"]*admin[^"]*"[^>]*>\s*Admin\s*</a>', r'<a href="research_admin.html" style="color:#ffcc80; font-weight:bold; text-decoration:none;" title="Anchor Engine Admin">Admin</a>', nav_html, flags=re.IGNORECASE)
    if 'research_admin.html' not in nav_html:
        nav_html = re.sub(r'<a[^>]*>\s*Admin\s*</a>', r'<a href="research_admin.html" style="color:#ffcc80; font-weight:bold; text-decoration:none;" title="Anchor Engine Admin">Admin</a>', nav_html, flags=re.IGNORECASE)

    media_sub_header = ""
    if media_links:
        media_sub_header = f"""
        <div class="no-print" style="background:#004d40; border-top:1px solid rgba(255,255,255,0.1); padding:12px 15px; border-bottom: 2px solid #e65100;">
            <div style="display:flex; flex-wrap:wrap; align-items:center; justify-content:center; gap:12px; max-width:1400px; margin:0 auto; font-family:sans-serif;">
                <span style="color:#ffcc80; font-size:13px; font-weight:bold; text-transform:uppercase; letter-spacing:1px; margin-right:5px; text-shadow:0 1px 2px rgba(0,0,0,0.2);">🎙️ AI Media Hub</span>
                {''.join(media_links)}
            </div>
        </div>
        """
    nav_html = nav_html + media_sub_header

    timestamp = datetime.now(est).strftime("%B %d, %Y %I:%M %p %Z")
    num_matches = len(df_valid)

    # Using the Banner Plate
    stats_bar_full = ANCHOR_PAYLOADS.get('plate_warning_banner', '') + f'<div style="background:#f4f4f4;border-top:1px solid #ddd;border-bottom:1px solid #ddd;font-family:sans-serif;font-size:14px;color:#555;padding:10px 15px;text-align:center;margin-bottom:0;"><strong>Study Data Current As Of:</strong> {timestamp} | <strong>Total Participants:</strong> {num_participants} | <strong>Total Autosomal matches:</strong> {num_matches:,}</div>'

    current_year = str(datetime.now(est).year)
    LEGAL_FOOTER = r'''<div class="legal-footer no-print" style="margin-top:50px;padding:20px;background:#f4f4f4;border-top:1px solid #ddd;text-align:center;color:#666;font-family:sans-serif;font-size:0.85em;clear:both;"><p style="margin-bottom:5px;font-size:1.1em;color:#333;"><strong>&copy; __YEAR__ Ronald Eugene Yates. All Rights Reserved.</strong></p><p style="margin-bottom:5px;">Generated by <em>The Forensic Genealogy Publisher&trade;</em></p><p style="font-style:italic;color:#888;margin-bottom:0;max-width:800px;margin-left:auto;margin-right:auto;">The terms "Forensic Handshake", "Collateral Saturation", and "ANCHOR" are trademarks of Ronald Eugene Yates.</p></div>'''.replace('__YEAR__', current_year)

    REGISTER_SEARCH_BOX = """
    <div class="no-print" style="margin: 20px auto; max-width: 1100px; background: #e0f2f1; padding: 25px; border-radius: 8px; border-left: 6px solid #004d40; box-shadow: 0 4px 15px rgba(0,0,0,0.05); display: flex; flex-direction: column; align-items: center; justify-content: center; font-family: sans-serif;">
        <div style="color: #004d40; font-weight: bold; font-size: 18px; margin-bottom: 15px; text-transform: uppercase; letter-spacing: 1px;">Live Registry Search</div>
        <input type="text" id="register-search" placeholder="🔍 Type a participant name, kit number, ancestor, or cM value to filter..."
               style="width: 100%; max-width: 800px; padding: 15px 25px; border: 2px solid #b2ebf2; border-radius: 30px; font-size: 16px; color: #004d40; outline: none; box-shadow: inset 0 2px 4px rgba(0,0,0,0.05); transition: all 0.3s;"
               onfocus="this.style.borderColor='#004d40'; this.style.boxShadow='0 4px 12px rgba(0,77,64,0.15)';"
               onblur="this.style.borderColor='#b2ebf2'; this.style.boxShadow='inset 0 2px 4px rgba(0,0,0,0.05)';"
               onkeyup="filterRegisterTable()">
    </div>
    <script>
    function filterRegisterTable() {
        let input = document.getElementById('register-search').value.toLowerCase();
        let rows = document.querySelectorAll('table#reg-table tbody tr');
        rows.forEach(row => {
            let text = row.textContent.toLowerCase();
            if(text.includes(input)) {
                row.style.display = '';
            } else {
                row.style.display = 'none';
            }
        });
    }
    </script>
    """

    def render_master(title, content, nav_b, bar, extra_css=""):
        s_info = REGISTER_SEARCH_BOX if nav_b else ""
        content_with_db = content.replace('__DATABASE_SCRIPT__', db_script_tag)
        return master_jinja.render(
            title=title, stats_bar=bar, nav_html=nav_html,
            site_info=s_info, main_content=content_with_db, extra_css=extra_css,
            legal_footer=LEGAL_FOOTER, js_core=SITE_TEMPLATES.get('js_core', ''), btt_btn=SITE_TEMPLATES.get('btt_btn', '')
        )

    def clean_lexicon(raw_html):
        cleaned = raw_html.replace('Biological Proof', 'Biological Validation')
        cleaned = cleaned.replace('biological proof', 'biological validation')
        cleaned = cleaned.replace('Proof-Grade', 'Validation-Grade')
        cleaned = cleaned.replace('proof-grade', 'validation-grade')
        cleaned = cleaned.replace('Integrity Score', 'Validation Status')
        cleaned = cleaned.replace('ranking metric', 'Match Strength')
        cleaned = cleaned.replace('Score Ranges', 'Confidence Levels')
        cleaned = re.sub(r'\bNone\s*\(I(\d+)\)', r'Unnamed Ancestor (I\1)', cleaned)
        return cleaned

    admin_tbody_html = ""
    for r in admin_rows:
        count_style = "color:#c62828; font-weight:bold;" if r['count'] == 0 else "color:#333;"
        admin_tbody_html += f"""<tr data-name="{r['sort']}" data-count="{r['count']}">
            <td style="text-align:left; padding-left:20px;">
                <strong style="color:#00838f; font-size:16px;">{r['name']}</strong> <span style="color:#777; font-size:12px;">(ID: {r['id']})</span><br>
                <span style="font-size:12px; color:#555;">Code: {r['code']} | Token: {r['person_key']}</span>
            </td>
            <td style="font-size:18px; {count_style}">{r['count']}</td>
        </tr>"""

    zm_tbody_html = ""
    if len(zero_rows) == 0: zm_tbody_html = '<tr><td colspan="3" style="text-align:center; padding:40px; color:#2e7d32; font-weight:bold;">All registered participants have matches. No zero-match participants found.</td></tr>'
    else:
        for r in zero_rows:
            zm_tbody_html += f"""<tr>
                <td style="text-align:left; padding-left:20px;"><strong style="color:#b71c1c; font-size:16px;">{r['name']}</strong></td>
                <td style="text-align:left;"><span style="color:#555; font-size:14px;">ID: {r['id']}<br>Code: {r['code']} | Token: {r['person_key']}</span></td>
                <td style="text-align:center; color:#c62828; font-weight:bold; font-size:18px;">0</td>
            </tr>"""

    print("    [+] Mass-Assembling Pages via Jinja2...")

    html_reg_za = df_reg_za.to_html(columns=["Participant Matches & Lineages"], index=False, border=1, classes="dataframe sortable", escape=False, table_id="reg-table")
    html_reg_az = df_reg_az.to_html(columns=["Participant Matches & Lineages"], index=False, border=1, classes="dataframe sortable", escape=False, table_id="reg-table")
    html_reg_match = df_reg_match.to_html(columns=["Participant Matches & Lineages"], index=False, border=1, classes="dataframe sortable", escape=False, table_id="reg-table")

    html_tree_za = df_tree_za[["Visual Lineage Path"]].to_html(index=False, border=1, classes="dataframe sortable", escape=False, table_id="reg-table")
    html_tree_az = df_tree_az[["Visual Lineage Path"]].to_html(index=False, border=1, classes="dataframe sortable", escape=False, table_id="reg-table")
    html_tree_match = df_tree_match[["Visual Lineage Path"]].to_html(index=False, border=1, classes="dataframe sortable", escape=False, table_id="reg-table")

    glossary_content = SITE_TEMPLATES.get('glossary_inline', '').replace('<details open ', '<details ').replace('<details open>', '<details>')

    pe_content = SITE_TEMPLATES.get('proof_engine_tmpl', '').replace('__CSS_BASE__', SITE_TEMPLATES.get('css_base', '')).replace('__STATS_BAR__', stats_bar_full).replace('__NAV_HTML__', nav_html).replace('__DATABASE_SCRIPT__', db_script_tag).replace('__LEGAL_FOOTER__', LEGAL_FOOTER)
    pages_to_upload["proof_engine.html"] = pe_content

    dossier_content = SITE_TEMPLATES.get('dossier_tmpl', '').replace('__CSS_BASE__', SITE_TEMPLATES.get('css_base', '')).replace('__STATS_BAR__', stats_bar_full).replace('__NAV_HTML__', nav_html).replace('__DATABASE_SCRIPT__', db_script_tag).replace('__LEGAL_FOOTER__', LEGAL_FOOTER)
    pages_to_upload["dna_dossier.html"] = dossier_content

    admin_content = ANCHOR_PAYLOADS.get('plate_velvet_rope', '') + """
    <style>
    .dash-grid { display: grid; grid-template-columns: repeat(auto-fit, minmax(200px, 1fr)); gap: 20px; max-width: 1000px; margin: 30px auto; font-family: sans-serif; }
    .dash-btn { background: white; border: 2px solid #ddd; border-radius: 8px; padding: 25px; text-align: center; text-decoration: none; color: #006064; font-weight: bold; font-size: 16px; transition: all 0.2s; box-shadow: 0 2px 4px rgba(0,0,0,0.05); }
    .dash-btn:hover { transform: translateY(-3px); box-shadow: 0 6px 12px rgba(0,0,0,0.1); border-color:#00838f; }
    .dash-btn.active-rpt { border-color: #ab47bc; background: #f3e5f5; color: #4a148c; }
    .dash-btn.ged-btn { border-color: #81d4fa; background: #e1f5fe; color: #01579b; }
    .dash-btn.anc-btn { border-color: #ffccbc; background: #fbe9e7; color: #bf360c; }
    .dash-btn.csv-btn { border-color: #ef9a9a; background: #ffebee; color: #c62828; }
    .dash-btn.zm-btn { border-color: #d32f2f; background: #ffebee; color: #b71c1c; }
    .dash-icon { font-size: 32px; display: block; margin-bottom: 10px; }
    .admin-header { border-bottom: 2px solid #004d40; padding-bottom: 15px; color: #004d40; max-width: 1000px; margin: 40px auto 10px auto; font-family: 'Georgia', serif; line-height: 1.6; font-size: 20px; }
    .sort-btns { text-align: center; margin-bottom: 20px; }
    .sort-btn { padding: 10px 20px; margin: 0 5px; border: none; border-radius: 4px; font-weight: bold; cursor: pointer; font-size: 14px; transition: all 0.2s; }
    .sort-btn.active { background: #004d40; color: white; }
    .sort-btn.inactive { background: #e0f2f1; color: #006064; }
    .sort-btn:hover { opacity: 0.9; }
    .admin-table { width: 100%; max-width: 1000px; margin: 0 auto; border-collapse: collapse; font-family: sans-serif; box-shadow: 0 2px 10px rgba(0,0,0,0.05); background: white; }
    .admin-table th { background: #004d40; color: white; padding: 15px; text-align: center; font-size: 15px; }
    .admin-table td { padding: 15px; border-bottom: 1px solid #ddd; text-align: center; vertical-align: middle; }
    .admin-table tr:hover { background-color: #f1f8e9; }
    </style>
    <div class="dash-grid no-print">
        <a href="engine_database.csv?v=__CACHE__" class="dash-btn" style="border-color:#4caf50; background:#e8f5e9; color:#2e7d32;" download><span class="dash-icon">&#128229;</span>Download CSV DB</a>
        <a href="match_to_unmasked.csv?v=__CACHE__" class="dash-btn" style="border-color:#1976d2; background:#e3f2fd; color:#1565c0;" download><span class="dash-icon">&#128193;</span>Download Master CSV</a>
        <a href="dna_report.html" class="dash-btn active-rpt"><span class="dash-icon">&#127891;</span>Report</a>
        <a href="proof_engine.html" class="dash-btn"><span class="dash-icon">&#128300;</span>Proof Engine</a>
        <a href="dna_dossier.html" class="dash-btn"><span class="dash-icon">&#128193;</span>Forensic Dossier</a>
        <a href="gedmatch_hub.html" class="dash-btn ged-btn"><span class="dash-icon">&#129516;</span>GEDmatch Hub</a>
        <a href="anchor_matrix.html" class="dash-btn anc-btn"><span class="dash-icon">&#9875;</span>Anchor Matrix</a>
        <a href="zero_matches.html" class="dash-btn zm-btn"><span class="dash-icon">&#128683;</span>Zero Matches</a>
        <a href="https://analytics.google.com/analytics/web/#/p342112997/reports/intelligenthome" target="_blank" class="dash-btn" style="border-color:#fb8c00; background:#fff3e0; color:#e65100;"><span class="dash-icon">&#128225;</span>Live Telemetry</a>
    </div>
    <h2 class="admin-header">Report - __VALID_COUNT__ Participants with matches</h2>
    <p style="text-align:center; margin-top:5px; margin-bottom:20px; color:#555; font-family:sans-serif;">
        <span style="color:#c62828; font-weight:bold;">__ZERO_COUNT__</span> participants currently report <a href="zero_matches.html" style="color:#c62828; text-decoration:underline;">zero matches</a>.
    </p>
    <div class="sort-btns no-print">
        <button id="btn-sort-az" class="sort-btn active" onclick="sortAdminTable('az')">Sort A-Z (Default)</button>
        <button id="btn-sort-count" class="sort-btn inactive" onclick="sortAdminTable('count')">Sort by Match Count (Low to High)</button>
    </div>
    <div class="table-scroll-wrapper" style="overflow-x:auto;">
        <table class="admin-table">
            <thead><tr><th style="width: 70%; text-align:left; padding-left:20px;">Participant Details</th><th style="width: 30%;">Total Matches</th></tr></thead>
            <tbody id="admin-tbody">
                __ADMIN_TBODY__
            </tbody>
        </table>
    </div>
    __DB_SCRIPT_TAG__
    <script>
    function sortAdminTable(mode) {
        let tbody = document.getElementById('admin-tbody');
        let rows = Array.from(tbody.querySelectorAll('tr'));
        rows.sort((a, b) => {
            if (mode === 'az') return a.getAttribute('data-name').localeCompare(b.getAttribute('data-name'));
            return parseInt(a.getAttribute('data-count')) - parseInt(b.getAttribute('data-count'));
        });
        tbody.innerHTML = '';
        rows.forEach(r => tbody.appendChild(r));
        document.getElementById('btn-sort-az').className = mode === 'az' ? 'sort-btn active' : 'sort-btn inactive';
        document.getElementById('btn-sort-count').className = mode === 'count' ? 'sort-btn active' : 'sort-btn inactive';
    }
    </script>
    """.replace('__CACHE__', str(cache_buster)).replace('__VALID_COUNT__', str(num_participants - len(zero_rows))).replace('__ZERO_COUNT__', str(len(zero_rows))).replace('__ADMIN_TBODY__', admin_tbody_html).replace('__DB_SCRIPT_TAG__', db_script_tag)

    zero_matches_content = ANCHOR_PAYLOADS.get('plate_velvet_rope', '') + """
    <style>
    .zm-header { border-bottom: 2px solid #b71c1c; padding-bottom: 10px; color: #b71c1c; max-width: 1000px; margin: 40px auto 10px auto; font-family: 'Georgia', serif; }
    .zm-subtitle { max-width: 1000px; margin: 0 auto 30px auto; font-family: sans-serif; color: #555; font-size: 15px; line-height: 1.6; }
    .zm-table { width: 100%; max-width: 1000px; margin: 0 auto; border-collapse: collapse; font-family: sans-serif; box-shadow: 0 2px 10px rgba(0,0,0,0.05); background: white; }
    .zm-table th { background: #b71c1c; color: white; padding: 15px; text-align: left; font-size: 15px; }
    .zm-table td { padding: 15px; border-bottom: 1px solid #ddd; text-align: left; vertical-align: middle; }
    .zm-table tr:hover { background-color: #ffebee; }
    </style>
    <div class="no-print" style="max-width: 1000px; margin: 20px auto;">
        <a href="research_admin.html" style="color: #006064; text-decoration: none; font-family: sans-serif; font-weight: bold;">&#8592; Back to Admin Hub</a>
    </div>
    <h2 class="zm-header">Zero Match Audit</h2>
    <p class="zm-subtitle">
        The following participants are registered in the study's authority source but currently report <strong style="color:#c62828;">0</strong> correlated matches in the main engine database.
        <br><span style="display:inline-block; margin-top:10px; padding:5px 10px; background:#ffebee; color:#b71c1c; font-weight:bold; border-radius:4px; border:1px solid #ffcdd2;">Total Zero-Match Participants: __ZERO_COUNT__</span>
    </p>
    <div class="table-scroll-wrapper" style="overflow-x:auto;">
        <table class="zm-table">
            <thead><tr><th style="width: 45%; padding-left:20px;">Participant Name</th><th style="width: 35%;">Participant Details</th><th style="width: 20%; text-align:center;">Match Count</th></tr></thead>
            <tbody>__ZM_TBODY__</tbody>
        </table>
    </div>
    __DB_SCRIPT_TAG__
    """.replace('__ZERO_COUNT__', str(len(zero_rows))).replace('__ZM_TBODY__', zm_tbody_html).replace('__DB_SCRIPT_TAG__', db_script_tag)

    anchor_matrix_content = ANCHOR_PAYLOADS.get('plate_velvet_rope', '') + """
    <style>
    .matrix-table { width: 100%; max-width: 1200px; margin: 0 auto; border-collapse: collapse; font-family: sans-serif; box-shadow: 0 2px 10px rgba(0,0,0,0.05); background: white; }
    .matrix-table th { background: #004d40; color: white; padding: 15px; text-align: center; font-size: 14px; text-transform: uppercase; letter-spacing: 1px; vertical-align: middle; }
    .matrix-table td { padding: 15px; border-bottom: 1px solid #eee; text-align: center; vertical-align: middle; }
    .matrix-table tr:hover { background-color: #f1f8e9; }
    .matrix-table td:first-child { text-align: left; }
    </style>
    <div class="no-print" style="max-width: 1200px; margin: 20px auto;">
        <a href="research_admin.html" style="color: #006064; text-decoration: none; font-family: sans-serif; font-weight: bold;">&#8592; Back to Admin Hub</a>
    </div>
    <div style="max-width:1200px; margin:0 auto; padding:20px; font-family:sans-serif; background:white; border-radius:8px; border-top: 5px solid #004d40; box-shadow: 0 4px 15px rgba(0,0,0,0.05);">
        <h2 style="color:#004d40; margin-top:0; display:flex; justify-content:space-between; align-items:center;">
            Forensic Validation Matrix
            <span style="font-size:12px; background:#fff3e0; color:#e65100; padding:4px 10px; border-radius:12px; border:1px solid #ffcc80;">Live Experimentation Mode Active</span>
        </h2>
        <p style="color:#555; font-size:15px; line-height:1.6;">
            This matrix dynamically evaluates each ancestral lineage using the dual-verification <strong>ANCHOR Framework</strong>.
            Adjust the forensic thresholds below to dynamically recalculate network strengths.
        </p>

        <div style="background:#f4f8f9; padding:20px; border-radius:6px; border:1px solid #b2ebf2; margin-bottom:25px; display:flex; gap:20px; align-items:center; justify-content:center; flex-wrap:wrap; font-size:14px; color:#006064;">
            <div style="font-weight:bold; letter-spacing:1px; text-transform:uppercase;">CSS v2a Parameters:</div>
            <label style="display:flex; align-items:center; gap:8px;">Min Proper Matches: <input type="number" id="cfg_pm" style="width:60px; padding:6px; border:1px solid #ccc; border-radius:4px; text-align:center; font-weight:bold;" value="2"></label>
            <label style="display:flex; align-items:center; gap:8px;">Min Vol (cM): <input type="number" id="cfg_vol" style="width:70px; padding:6px; border:1px solid #ccc; border-radius:4px; text-align:center; font-weight:bold;" value="40"></label>
            <label style="display:flex; align-items:center; gap:8px;">Min Collateral Lines: <input type="number" id="cfg_col" style="width:60px; padding:6px; border:1px solid #ccc; border-radius:4px; text-align:center; font-weight:bold;" value="2"></label>
            <button onclick="applyForensicConfig()" style="padding:8px 20px; background:#00838f; color:white; border:none; border-radius:4px; cursor:pointer; font-weight:bold; transition:background 0.2s;">Apply</button>
            <button onclick="resetForensicConfig()" style="padding:8px 20px; background:#cfd8dc; color:#37474f; border:none; border-radius:4px; cursor:pointer; font-weight:bold;">Reset</button>
        </div>

        <div class="table-scroll-wrapper" style="overflow-x:auto;">
            <table class="matrix-table">
                <thead>
                    <tr>
                        <th>Emerged Ancestor</th>
                        <th>Genetic Evidence<br><span style="font-size:11px; color:#cce5ff; font-weight:normal;">(CSS v2a)</span></th>
                        <th>Documentary Evidence<br><span style="font-size:11px; color:#cce5ff; font-weight:normal;">(DOCS)</span></th>
                        <th>ANCHOR Score</th>
                    </tr>
                </thead>
                <tbody id="dynamic-matrix-body">
                </tbody>
            </table>
        </div>
    </div>
    __DB_SCRIPT_TAG__
    <script>
        document.getElementById('cfg_pm').value = localStorage.getItem('ANC_CFG_PM') || 2;
        document.getElementById('cfg_vol').value = localStorage.getItem('ANC_CFG_VOL') || 40;
        document.getElementById('cfg_col').value = localStorage.getItem('ANC_CFG_COL') || 2;

        function applyForensicConfig() {
            localStorage.setItem('ANC_CFG_PM', document.getElementById('cfg_pm').value);
            localStorage.setItem('ANC_CFG_VOL', document.getElementById('cfg_vol').value);
            localStorage.setItem('ANC_CFG_COL', document.getElementById('cfg_col').value);
            renderMatrix();
        }

        function resetForensicConfig() {
            localStorage.setItem('ANC_CFG_PM', 2);
            localStorage.setItem('ANC_CFG_VOL', 40);
            localStorage.setItem('ANC_CFG_COL', 2);
            document.getElementById('cfg_pm').value = 2;
            document.getElementById('cfg_vol').value = 40;
            document.getElementById('cfg_col').value = 2;
            renderMatrix();
        }

        function renderMatrix() {
            if(!window.DB) return;
            let tbody = document.getElementById('dynamic-matrix-body');
            tbody.innerHTML = '<tr><td colspan="4" style="text-align:center; padding:30px; color:#006064; font-weight:bold;">Analyzing Autosomal Network Engine...</td></tr>';

            let t_pm = parseInt(localStorage.getItem('ANC_CFG_PM')) || 2;
            let t_vol = parseFloat(localStorage.getItem('ANC_CFG_VOL')) || 40.0;
            let t_col = parseInt(localStorage.getItem('ANC_CFG_COL')) || 2;

            setTimeout(() => {
                let groups = {};
                window.DB.forEach(r => {
                    let label = r.Dir_Label || r.Authority_Directory_Label;
                    if(!label || label === 'No Matches') return;
                    let cleanLabel = label.split('(')[0].trim();
                    if(!groups[cleanLabel]) groups[cleanLabel] = { pm:0, vol:0, testers: new Set(), lineage: r.Lineage||"" };
                    groups[cleanLabel].pm += 1;
                    groups[cleanLabel].vol += (parseFloat(String(r.cM).replace(/[^0-9.]/g, '')) || 0);
                    groups[cleanLabel].testers.add(r.Participant_Code || r.Tester_Code);
                });

                tbody.innerHTML = '';
                Object.keys(groups).sort().forEach(label => {
                    let g = groups[label];
                    let cssPass = (g.pm >= t_pm && g.vol >= t_vol && g.testers.size >= t_col);
                    let gd = g.lineage ? g.lineage.split('->').length : 0;

                    let cssBadge = cssPass ?
                        "<span style='color:#155724; background:#d4edda; border:1px solid #c3e6cb; padding:4px 8px; border-radius:4px; font-weight:bold; font-size:11px;'>CSS: PASS</span>" :
                        "<span style='color:#721c24; background:#f8d7da; border:1px solid #f5c6cb; padding:4px 8px; border-radius:4px; font-weight:bold; font-size:11px;'>CSS: FAIL</span>";
                    let docsBadge = "<span style='color:#004085; background:#cce5ff; border:1px solid #b8daff; padding:4px 8px; border-radius:4px; font-weight:bold; font-size:11px;'>DOCS: EVAL</span>";

                    let pmPct = Math.round((g.pm / t_pm) * 100);
                    let volPct = Math.round((g.vol / t_vol) * 100);
                    let colPct = Math.round((g.testers.size / t_col) * 100);

                    let cssHtml = `<div style='font-size:13px; color:#444; margin-bottom:8px; display:flex; justify-content:center; gap:8px; flex-wrap:wrap;'>
                        <span style="background:#e8f5e9; padding:4px 8px; border-radius:4px; border:1px solid #c8e6c9;"><b>PM:</b> ${g.pm} <span style="color:#2e7d32; font-size:11px; font-weight:bold;">(${pmPct}%)</span></span>
                        <span style="background:#e8f5e9; padding:4px 8px; border-radius:4px; border:1px solid #c8e6c9;"><b>Vol:</b> ${g.vol.toFixed(1)}cM <span style="color:#2e7d32; font-size:11px; font-weight:bold;">(${volPct}%)</span></span>
                        <span style="background:#e8f5e9; padding:4px 8px; border-radius:4px; border:1px solid #c8e6c9;"><b>Col:</b> ${g.testers.size} <span style="color:#2e7d32; font-size:11px; font-weight:bold;">(${colPct}%)</span></span>
                    </div>${cssBadge}`;

                    let docsHtml = "<div style='font-size:13px; color:#444; margin-bottom:8px;'><b>AX:</b> Yes &nbsp;|&nbsp; <b>GD:</b> " + gd + " &nbsp;|&nbsp; <b>Vts:</b> Pend &nbsp;|&nbsp; <b>CC:</b> Pend &nbsp;|&nbsp; <b>TP:</b> Yes</div>" + docsBadge;

                    let anchorStatus = cssPass ?
                        "<span style='color:white; background:#2e7d32; padding:6px 12px; border-radius:20px; font-weight:bold; font-size:13px; box-shadow:0 2px 4px rgba(0,0,0,0.1);'>🟢 Confirmed</span>" :
                        "<span style='color:white; background:#c62828; padding:6px 12px; border-radius:20px; font-weight:bold; font-size:13px; box-shadow:0 2px 4px rgba(0,0,0,0.1);'>🔴 Speculative</span>";

                    let tr = document.createElement('tr');
                    tr.innerHTML = '<td style="text-align:left;"><strong style="color:#006064; font-size:15px;">' + label + '</strong></td><td>' + cssHtml + '</td><td>' + docsHtml + '</td><td>' + anchorStatus + '</td>';
                    tbody.appendChild(tr);
                });
            }, 100);
        }
        window.addEventListener('load', renderMatrix);
    </script>
    """.replace('__DB_SCRIPT_TAG__', db_script_tag)

    pages_to_upload["research_admin.html"] = render_master("Yates Research Admin Hub", admin_content, False, stats_bar_full, SITE_TEMPLATES.get('admin_css', ''))
    pages_to_upload["zero_matches.html"] = render_master("Zero Matches Audit", zero_matches_content, False, stats_bar_full, "")
    pages_to_upload["anchor_matrix.html"] = render_master("Forensic Validation Matrix", anchor_matrix_content, False, stats_bar_full, "")
    pages_to_upload["about.shtml"] = render_master("Origin of the Methodology", ANCHOR_PAYLOADS.get('plate_about', ''), False, stats_bar_full, "")

    anchor_injected = SITE_TEMPLATES.get('anchor_content', '').replace('__TAB_FRAMEWORK__', SITE_TEMPLATES.get('anchor_theory', ''))
    anchor_injected = anchor_injected + ANCHOR_PAYLOADS.get('plate_methodology_anchor', '')
    pages_to_upload["anchor.html"] = render_master("ANCHOR Database", anchor_injected, False, stats_bar_full, SITE_TEMPLATES.get('anchor_css', ''))
    pages_to_upload["dna_network.html"] = render_master("DNA Network Visualizer", SITE_TEMPLATES.get('network_page', ''), False, stats_bar_full, "")

    # 🌟 INJECT SPLITTER ONLY INTO CONSOLIDATOR 🌟
    consol_html = SITE_TEMPLATES.get('consolidator_html', '') + SITE_TEMPLATES.get('consolidator_js', '') + ANCHOR_PAYLOADS.get('report_splitter_js', '')
    consol_html = consol_html.replace('All Fails Net (Auto-Gen)', 'IGNORED_PRESET')
    pages_to_upload["proof_consolidator.html"] = render_master("Master Proof Report", consol_html, False, stats_bar_full, SITE_TEMPLATES.get('consolidator_css', ''))

    base_style = "text-align:center; margin:15px auto; max-width:1100px; padding:10px; background:#e0f7fa; border:1px solid #b2ebf2; border-radius:4px; font-family:sans-serif; font-size:14px;"
    active_style = "color:#006064; font-weight:bold;"
    link_style = "color:#00acc1; text-decoration:none;"

    toggle_reg_za = f'<div class="no-print" style="{base_style}"><strong>Sort Register:</strong> &nbsp;<span style="{active_style}">By Ancestral Line (Z-A)</span> &nbsp;|&nbsp; <a href="ons_yates_dna_register_participants.shtml" style="{link_style}">By Participant (A-Z)</a> &nbsp;|&nbsp; <a href="ons_yates_dna_register_matches.shtml" style="{link_style}">By Match (A-Z)</a></div>'
    toggle_reg_az = f'<div class="no-print" style="{base_style}"><strong>Sort Register:</strong> &nbsp;<a href="ons_yates_dna_register.shtml" style="{link_style}">By Ancestral Line (Z-A)</a> &nbsp;|&nbsp; <span style="{active_style}">By Participant (A-Z)</span> &nbsp;|&nbsp; <a href="ons_yates_dna_register_matches.shtml" style="{link_style}">By Match (A-Z)</a></div>'
    toggle_reg_match = f'<div class="no-print" style="{base_style}"><strong>Sort Register:</strong> &nbsp;<a href="ons_yates_dna_register.shtml" style="{link_style}">By Ancestral Line (Z-A)</a> &nbsp;|&nbsp; <a href="ons_yates_dna_register_participants.shtml" style="{link_style}">By Participant (A-Z)</a> &nbsp;|&nbsp; <span style="{active_style}">By Match (A-Z)</span></div>'
    toggle_tree_za = f'<div class="no-print" style="{base_style}"><strong>Sort Trees:</strong> &nbsp;<span style="{active_style}">By Ancestral Line (Z-A)</span> &nbsp;|&nbsp; <a href="just-trees-az.shtml" style="{link_style}">By Participant (A-Z)</a> &nbsp;|&nbsp; <a href="just-trees-matches.shtml" style="{link_style}">By Match (A-Z)</a></div>'
    toggle_tree_az = f'<div class="no-print" style="{base_style}"><strong>Sort Trees:</strong> &nbsp;<a href="just-trees.shtml" style="{link_style}">By Ancestral Line (Z-A)</a> &nbsp;|&nbsp; <span style="{active_style}">By Participant (A-Z)</span> &nbsp;|&nbsp; <a href="just-trees-matches.shtml" style="{link_style}">By Match (A-Z)</a></div>'
    toggle_tree_match = f'<div class="no-print" style="{base_style}"><strong>Sort Trees:</strong> &nbsp;<a href="just-trees.shtml" style="{link_style}">By Ancestral Line (Z-A)</a> &nbsp;|&nbsp; <a href="just-trees-az.shtml" style="{link_style}">By Participant (A-Z)</a> &nbsp;|&nbsp; <span style="{active_style}">By Match (A-Z)</span></div>'

    pages_to_upload["ons_yates_dna_register.shtml"] = render_master("ONS Yates Study DNA Register", ANCHOR_PAYLOADS['table_cleanup_css'] + toggle_reg_za + f'<div class="table-scroll-wrapper">{html_reg_za}</div>', True, stats_bar_full, SITE_TEMPLATES.get('register_css', ''))
    pages_to_upload["ons_yates_dna_register_participants.shtml"] = render_master("ONS Yates Study DNA Register", ANCHOR_PAYLOADS['table_cleanup_css'] + toggle_reg_az + f'<div class="table-scroll-wrapper">{html_reg_az}</div>', True, stats_bar_full, SITE_TEMPLATES.get('register_css', ''))
    pages_to_upload["ons_yates_dna_register_matches.shtml"] = render_master("ONS Yates Study DNA Register", ANCHOR_PAYLOADS['table_cleanup_css'] + toggle_reg_match + f'<div class="table-scroll-wrapper">{html_reg_match}</div>', True, stats_bar_full, SITE_TEMPLATES.get('register_css', ''))

    pages_to_upload["just-trees.shtml"] = render_master("Ancestor Register (Trees View)", ANCHOR_PAYLOADS['table_cleanup_css'] + toggle_tree_za + f'<div class="table-scroll-wrapper">{html_tree_za}</div>', True, stats_bar_full, SITE_TEMPLATES.get('register_css', ''))
    pages_to_upload["just-trees-az.shtml"] = render_master("Ancestor Register (Trees View)", ANCHOR_PAYLOADS['table_cleanup_css'] + toggle_tree_az + f'<div class="table-scroll-wrapper">{html_tree_az}</div>', True, stats_bar_full, SITE_TEMPLATES.get('register_css', ''))
    pages_to_upload["just-trees-matches.shtml"] = render_master("Ancestor Register (Trees View)", ANCHOR_PAYLOADS['table_cleanup_css'] + toggle_tree_match + f'<div class="table-scroll-wrapper">{html_tree_match}</div>', True, stats_bar_full, SITE_TEMPLATES.get('register_css', ''))

    pages_to_upload["data_glossary.shtml"] = render_master("Data Glossary", glossary_content, False, stats_bar_full, "")
    pages_to_upload["gedmatch_integration.shtml"] = render_master("GEDmatch Hub", SITE_TEMPLATES.get('gedmatch_inline', ''), False, stats_bar_full, "")
    pages_to_upload["contents.shtml"] = render_master("Yates Study User Guide", SITE_TEMPLATES.get('contents_content', ''), False, stats_bar_full, "")
    pages_to_upload["share_dna.shtml"] = render_master("Share Your Ancestry DNA Matches", SITE_TEMPLATES.get('share_content', ''), False, stats_bar_full, "")
    pages_to_upload["subscribe.shtml"] = render_master("Join the Yates Research Community", SITE_TEMPLATES.get('subscribe_content', ''), False, stats_bar_full, "")
    pages_to_upload["dna_theory_of_the_case.htm"] = render_master("Theory of the Case", SITE_TEMPLATES.get('theory_page', ''), False, stats_bar_full, "")

    pages_to_upload["anchor_frame.htm"] = pages_to_upload["anchor.html"]
    pages_to_upload["dna_network.shtml"] = pages_to_upload["dna_network.html"]
    pages_to_upload["admin_singletons.shtml"] = pages_to_upload["ons_yates_dna_register.shtml"]
    pages_to_upload["admin_singletons_participants.shtml"] = pages_to_upload["ons_yates_dna_register_participants.shtml"]
    pages_to_upload["yates_ancestor_register.shtml"] = pages_to_upload["ons_yates_dna_register.shtml"]

    print("    [*] Scanning and updating old header links to the new Anchor directory...")
    for fn in pages_to_upload:
        pages_to_upload[fn] = pages_to_upload[fn].replace("https://yates.one-name.net/ons-study/", "https://yates.one-name.net/anchor/anc-1000-yates/")
        pages_to_upload[fn] = pages_to_upload[fn].replace("/ons-study/", "/anchor/anc-1000-yates/")
        pages_to_upload[fn] = pages_to_upload[fn].replace('href="ons-study/', 'href="anchor/anc-1000-yates/')

    if not os.path.exists('site'): os.makedirs('site')

    print("\n[LOCAL] Overwriting Files & Generating Enhanced Engine Database...")
    df_valid.to_csv(CSV_DB, index=False, encoding="utf-8")
    with open("site/database.js", "w", encoding="utf-8") as f: f.write(js_file_content)

    print("    [✨] Applying final lexicon scrub and injecting UI Upgraders...")
    for p_name, p_content in pages_to_upload.items():
        if p_name.endswith('.html') or p_name.endswith('.shtml') or p_name.endswith('.htm'):
            scrubbed = clean_lexicon(p_content)
            if '</body>' in scrubbed:
                scrubbed = scrubbed.replace('</body>', f"{ANCHOR_PAYLOADS.get('deep_path_js', '')}\n</body>")
            else:
                scrubbed += ANCHOR_PAYLOADS.get('deep_path_js', '')
            pages_to_upload[p_name] = scrubbed

    for fn, content in pages_to_upload.items():
        local_path = os.path.join('site', fn)
        if os.path.exists(local_path): os.remove(local_path)
        with open(local_path, "w", encoding="utf-8") as f: f.write(content)

    print("\n[STEP 3] Uploading via FTP to Live Server...")
    upload_success = 0; upload_fails = 0; failed_files = []

    try:
        ftps = FTP_TLS()
        ftps.connect(HOST, 21); ftps.auth(); ftps.login(USER, PASS); ftps.prot_p()
        ftps.cwd("anchor/anc-1000-yates")

        files_to_upload = list(pages_to_upload.keys()) + ["database.js"]
        if os.path.exists(CSV_DB):
            import shutil
            shutil.copyfile(CSV_DB, os.path.join('site', CSV_DB))
            files_to_upload.append(CSV_DB)

        for i, fn in enumerate(files_to_upload, 1):
            local_path = os.path.join('site', fn)
            try:
                with open(local_path, "rb") as fh: ftps.storbinary(f"STOR {fn}", fh)
                upload_success += 1; print(f"    [{i}/{len(files_to_upload)}] 📤 Success: {fn}")
            except Exception as file_e:
                upload_fails += 1; failed_files.append(fn); print(f"    [{i}/{len(files_to_upload)}] ❌ FAILED: {fn} ({file_e})")

        ftps.quit()
        finish_time = datetime.now(est).strftime("%I:%M:%S %p EST")
        print("\n" + "="*60)
        if upload_fails == 0:
            print(f"✅ PUBLISHER COMPLETE: All {upload_success} files successfully published to London at {finish_time}.")
        else:
            print(f"⚠️ PUBLISHER FINISHED WITH ERRORS: {upload_success} uploaded, {upload_fails} failed at {finish_time}.")
        print("="*60)

    except Exception as e: print(f"\n❌ CRITICAL FTP ERROR: Could not connect to the server. {e}")

run_publisher()

      [CELL 5B] PUBLISHER STARTING (Jinja2 Component Router)...
    [*] Connecting to FTP Server for Data Sweeps...
    [+] Fetching Central Master Template from London Hub...
    [+] Mass-Assembling Pages via Jinja2...
    [*] Scanning and updating old header links to the new Anchor directory...

[LOCAL] Overwriting Files & Generating Enhanced Engine Database...
    [✨] Applying final lexicon scrub and injecting UI Upgraders...

[STEP 3] Uploading via FTP to Live Server...
    [1/28] 📤 Success: proof_engine.html
    [2/28] 📤 Success: dna_dossier.html
    [3/28] 📤 Success: research_admin.html
    [4/28] 📤 Success: zero_matches.html
    [5/28] 📤 Success: anchor_matrix.html
    [6/28] 📤 Success: about.shtml
    [7/28] 📤 Success: anchor.html
    [8/28] 📤 Success: dna_network.html
    [9/28] 📤 Success: proof_consolidator.html
    [10/28] 📤 Success: ons_yates_dna_register.shtml
    [11/28] 📤 Success: ons_yates_dna_register_participants.shtml
    [12/28] 📤 Success: ons_yates_dna_register_mat